In [ ]:
import re 
import torch 
import torch.nn as nn 
import torch.optim as optim 
from collections import Counter 
from torch.utils.data import Dataset, DataLoader  
from transformers import AutoTokenizer, AutoModelForSequenceClassification 
from datasets import load_dataset 
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report 

In [12]:
raw = load_dataset("tasksource/jigsaw_toxicity")
print(raw)

DatasetDict({
    train: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 159571
    })
})


In [13]:
full = raw['train'] 

split_1 = full.train_test_split(test_size = 0.2, seed = 42)
train_raw = split_1['train']
temp = split_1['test']

split_2 = temp.train_test_split(test_size = 0.5, seed = 42)
val_raw = split_2['train']
test_raw = split_2['test']

print(f"Train: {len(train_raw)} | Validation: {len(val_raw)} | Test: {len(test_raw)}")

Train: 127656 | Validation: 15957 | Test: 15958


In [14]:
def clean_text(text):
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\s+' , ' ', text).strip()
    text = text.lower()
    return text

In [30]:
LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

def to_clean_df(hf_dataset):
    df = hf_dataset.to_pandas()
    df = df[df['comment_text'].str.len() > 20]
    df['comment_text'] = df['comment_text'].apply(clean_text)
    df['label'] = df[LABELS].max(axis=1)
    return df[['comment_text', 'label']]

In [31]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [32]:
class ToxicDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['comment_text'].reset_index(drop=True)
        self.labels = df['label'].reset_index(drop=True)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        enc = self.tokenizer(
            text, 
            truncation=True, 
            padding='max_length', 
            max_length=128, 
            return_tensors='pt'
        )

        return {
            'input_ids': enc['input_ids'].squeeze(0),        
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)  
        }

In [33]:
train_df = to_clean_df(train_raw)
val_df = to_clean_df(val_raw)
test_df = to_clean_df(test_raw)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

train_clean = ToxicDataset(train_df, tokenizer)
val_clean = ToxicDataset(val_df, tokenizer)
test_clean = ToxicDataset(test_df, tokenizer)

sample = train_clean[0]
print(sample['input_ids'].shape)
print(sample['label'])

Train: 127208 | Val: 15901 | Test: 15891
torch.Size([128])
tensor(0)


In [22]:
import os
print(os.getcwd())

c:\Users\ASUS\Documents\A-Machine Learning projects\toxic-comment-classifier\notebooks


In [23]:
import os

os.makedirs("../data/raw", exist_ok=True)
full.save_to_disk("../data/raw/jigsaw_toxicity")

Saving the dataset (1/1 shards): 100%|██████████| 159571/159571 [00:00<00:00, 1374832.91 examples/s]


In [34]:
train_df = to_clean_df(train_raw)
val_df = to_clean_df(val_raw)
test_df = to_clean_df(test_raw)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

train_clean = ToxicDataset(train_df, tokenizer)
val_clean = ToxicDataset(val_df, tokenizer)
test_clean = ToxicDataset(test_df, tokenizer)

sample = train_clean[0]
print(sample['input_ids'].shape)
print(sample['label'])

Train: 127208 | Val: 15901 | Test: 15891
torch.Size([128])
tensor(0)


In [35]:
import os

os.makedirs("../data/processed", exist_ok=True)

train_df.to_csv("../data/processed/train.csv", index=False)
val_df.to_csv("../data/processed/val.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)